# 📗 Logistic Regression — Kaggle Titanic Dataset pe Applied
### Atlas Skilltech University | Instructor Reference

---

**Dataset:** Titanic — Kya Yeh Passenger Survive Karega? (Kaggle Classic)
- Hum ise **seaborn built-in datasets se directly download** karenge (same data, Kaggle account ki zaroorat nahi)
- Ya phir download karo: https://www.kaggle.com/c/titanic

**Goal:** Predict karo ki ek passenger **survived (1)** hua ya **died (0)** — features jaise age, class, sex, etc. use karke.

**Structure (Notebook 1 jaise hi order):**
1. Dataset Load Karo
2. EDA (Exploratory Data Analysis)
3. Preprocessing (missing values handle karo, categories encode karo)
4. Train-Test Split aur Scaling
5. Model Training
6. Confusion Matrix aur Accuracy
7. Precision, Recall, F1 Score
8. ROC Curve aur AUC
9. Performance Summary
10. Multiclass Extension (Passenger Class Prediction)
11. Capstone Exercise

> 💡 Har mushkil keyword ko code se **pehle** markdown cell mein explain kiya gaya hai.

---
## 🔧 Step 0 — Libraries Import Karo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, roc_curve
)

print('Saari libraries successfully import ho gayi ✅')

---
## 📌 Step 1 — Dataset Load Karo

**Yeh keywords yaad rakho:**
- **seaborn** — kuch free built-in datasets hote hain jo ek line mein load ho jaate hain
- **Titanic dataset** — 891 passengers ka data, batata hai ki 1912 ke Titanic shipwreck mein kaun bacha

**Columns ka matlab:**
| Column | Matlab |
|---|---|
| survived | 0 = Nahi Bacha, 1 = Bacha (TARGET) |
| pclass | Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd |
| sex | Passenger ka gender |
| age | Umar saalon mein |
| sibsp | Saath mein kitne siblings/spouses the |
| parch | Saath mein kitne parents/children the |
| fare | Ticket ki kimat |
| embarked | Kaahan se chade (C/Q/S) |
| class | Ticket class text mein |
| who | man/woman/child |
| adult_male | True/False |
| deck | Cabin ka deck |
| embark_town | Kaahan se chade — sheher ka naam |
| alive | yes/no |
| alone | True agar koi family saath nahi |

In [ ]:
# Titanic dataset seaborn se load karte hain (Kaggle version jaisa hi)
df = sns.load_dataset('titanic')

print('Shape:', df.shape)
df.head()

In [ ]:
print('Column names:', df.columns.tolist())
print()
print('Data types (har column ka type):')
print(df.dtypes)

---
## 📌 Step 2 — EDA (Exploratory Data Analysis)

**Yeh keywords yaad rakho:**
- **EDA** — model banane se pehle charts aur statistics se data ko samajhna aur patterns dhundhna
- **Missing values (NaN)** — khaali cells; zyaadatar ML models inhe directly handle nahi kar sakte
- **isnull().sum()** — har column mein kitne missing values hain yeh count karta hai
- **value_counts()** — dikhata hai ki har unique value kitni baar aaya

In [ ]:
# Basic statistics
print('Basic Statistics (numbers ka summary):')
df.describe()

In [ ]:
# Missing values check
print('Har column mein Missing Values:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Class balance check — classification mein hamesha yeh check karo!
print('Survival counts (kitne bache, kitne nahi bache):')
print(df['survived'].value_counts())
print()
print('Survival rate (bachhne ka rate):', df['survived'].mean().round(3))

In [ ]:
# EDA Charts — 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# 1. Survival count
df['survived'].value_counts().plot(kind='bar', ax=axes[0,0], color=['salmon', 'steelblue'], edgecolor='black')
axes[0,0].set_title('Survival Count (Kitne Bache)')
axes[0,0].set_xticklabels(['Nahi Bacha (0)', 'Bacha (1)'], rotation=0)
axes[0,0].set_ylabel('Count')

# 2. Survival by Sex
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0,1], color=['steelblue', 'salmon'], edgecolor='black')
axes[0,1].set_title('Gender ke Hisaab se Bachhne ka Rate')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].set_xticklabels(['Female', 'Male'], rotation=0)

# 3. Survival by Passenger Class
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[0,2], color=['gold', 'silver', 'brown'], edgecolor='black')
axes[0,2].set_title('Passenger Class ke Hisaab se Bachhne ka Rate')
axes[0,2].set_ylabel('Survival Rate')
axes[0,2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

# 4. Age distribution by survival
df[df['survived']==1]['age'].dropna().hist(bins=20, ax=axes[1,0], alpha=0.7, color='steelblue', label='Bacha')
df[df['survived']==0]['age'].dropna().hist(bins=20, ax=axes[1,0], alpha=0.7, color='salmon', label='Nahi Bacha')
axes[1,0].set_title('Umar ka Distribution — Survival ke Hisaab se')
axes[1,0].set_xlabel('Umar')
axes[1,0].legend()

# 5. Fare distribution by survival
df[df['survived']==1]['fare'].hist(bins=20, ax=axes[1,1], alpha=0.7, color='steelblue', label='Bacha')
df[df['survived']==0]['fare'].hist(bins=20, ax=axes[1,1], alpha=0.7, color='salmon', label='Nahi Bacha')
axes[1,1].set_title('Ticket ki Kimat — Survival ke Hisaab se')
axes[1,1].set_xlabel('Fare (Ticket Kimat)')
axes[1,1].legend()

# 6. Survival by embarkation point
df.groupby('embark_town')['survived'].mean().plot(kind='bar', ax=axes[1,2], color='teal', edgecolor='black')
axes[1,2].set_title('Kaahan Se Chade — Bachhne ka Rate')
axes[1,2].set_ylabel('Survival Rate')
axes[1,2].tick_params(axis='x', rotation=15)

plt.suptitle('Titanic EDA — Data Explore Karna', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — sirf numeric columns ke liye
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features ke Beech Relation')
plt.tight_layout()
plt.show()

---
## 📌 Step 3 — Preprocessing (Data Saaf Karna)

**Yeh keywords yaad rakho:**
- **Imputation** — missing values ko kisi sensible value se bhar dena (mean, median, mode)
- **Median imputation** — missing values ko median (beech ki value) se bharna; outliers hone pe mean se better hai
- **Label Encoding** — text categories ko numbers mein convert karna (jaise male=1, female=0)
- **Feature selection** — kaun se columns use karne hain decide karna; irrelevant ya duplicate columns hatana
- **drop()** — DataFrame se ek column hatana

In [ ]:
# Useful columns select karte hain
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target   = 'survived'

df_clean = df[features + [target]].copy()

print('Cleaning se pehle — missing values:')
print(df_clean.isnull().sum())

In [ ]:
# 'age' ki missing values ko median se bharte hain
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)

# 'embarked' ki missing values ko sabse common value (mode) se bharte hain
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)

print('Cleaning ke baad — missing values:')
print(df_clean.isnull().sum())

In [ ]:
# Categorical columns encode karte hain
# sex: female=0, male=1
df_clean['sex'] = LabelEncoder().fit_transform(df_clean['sex'])

# embarked: C=0, Q=1, S=2
df_clean['embarked'] = LabelEncoder().fit_transform(df_clean['embarked'])

print('Encoding ke baad:')
print(df_clean.head())
print()
print('Data types:')
print(df_clean.dtypes)

---
## 📌 Step 4 — Train-Test Split aur Feature Scaling

**Yeh keywords yaad rakho:**
- **X** — feature matrix (saare input columns)
- **y** — target vector (jo hum predict karna chahte hain)
- **StandardScaler** — features ko transform karta hai taaki mean=0 aur standard deviation=1 ho
- **fit_transform** — scaling parameters seekho AUR apply karo (sirf training data pe use karo)
- **transform** — pehle se sikhe parameters apply karo (test data pe use karo)

In [ ]:
X = df_clean.drop('survived', axis=1)
y = df_clean['survived']

print('Feature columns:', X.columns.tolist())
print('X ka shape:', X.shape)
print('y ka shape:', y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y → train aur test dono mein same class balance rakhe

print(f'Train size : {len(X_train)}')
print(f'Test  size : {len(X_test)}')
print()
print('Train mein class balance:')
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling ho gayi ✅')

---
## 📌 Step 5 — Model Training

**Yeh keywords yaad rakho:**
- **solver** — woh algorithm jo best weights dhundne ke liye use hota hai; `lbfgs` chhote datasets ke liye default hai
- **max_iter** — solver ko solution dhundhne ke liye maximum iterations ki limit
- **C (regularisation)** — model ki complexity control karta hai; chota C = zyada regularisation = simpler model
- **Regularisation** — ek technique jo large coefficients ko penalise karke overfitting rokti hai

In [ ]:
model = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0, random_state=42)
model.fit(X_train_sc, y_train)

print('Model train ho gaya ✅')

# Feature importance (coefficients) dikhate hain
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0].round(4)
}).sort_values('Coefficient', key=abs, ascending=False)

print('\nFeature Coefficients (absolute value jitna bada, utna zyada important):')
print(coef_df)

In [ ]:
# Feature coefficients visualise karte hain
plt.figure(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'salmon' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient ki Value')
plt.title('Logistic Regression — Feature Coefficients\n(positive = survival probability badhta hai)')
plt.tight_layout()
plt.show()

---
## 📌 Step 6 — Confusion Matrix aur Accuracy

**Yaad karo:**
- **TP (True Positive)** — predict kiya Survived, actually Survived tha
- **TN (True Negative)** — predict kiya Died, actually Died tha
- **FP (False Positive / Type I Error)** — predict kiya Survived, actually Died tha
- **FN (False Negative / Type II Error)** — predict kiya Died, actually Survived tha
- **Accuracy** = (TP + TN) / Total

In [ ]:
y_pred      = model.predict(X_test_sc)
y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {acc:.4f} ({acc:.2%})')
print('\nConfusion Matrix (raw numbers):')
print(cm)

In [ ]:
# --- TARIKA 1 (purana): seaborn heatmap ---
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Nahi Bacha', 'Bacha'],
            yticklabels=['Nahi Bacha', 'Bacha'])
plt.xlabel('Predicted (Model ne kya bola)')
plt.ylabel('Actual (Actually kya tha)')
plt.title('Confusion Matrix — seaborn heatmap (purana tarika)')
plt.tight_layout()
plt.show()

In [ ]:
# --- TARIKA 2 (modern, recommended): ConfusionMatrixDisplay ---
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Nahi Bacha', 'Bacha'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — ConfusionMatrixDisplay (naya aur recommended tarika)')
plt.tight_layout()
plt.show()

---
## 📌 Step 7 — Precision, Recall, F1 Score

**Yaad karo:**
$$\text{Precision} = \frac{TP}{TP + FP}$$
$$\text{Recall} = \frac{TP}{TP + FN}$$
$$F_1 = 2 \times \frac{P \times R}{P + R}$$

Titanic ke context mein:
- **Precision** — model ne jitne survivors predict kiye, unme se actually kitne survive kiye?
- **Recall** — jo actually survive kiye the, unme se model ne kitno ko sahi identify kiya?

In [ ]:
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.4f}')
print()
print('Pura Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Nahi Bacha', 'Bacha']))

---
## 📌 Step 8 — ROC Curve aur AUC

**Yaad karo:**
- **ROC Curve** — True Positive Rate (sensitivity) aur False Positive Rate ke beech trade-off dikhata hai har threshold pe
- **AUC** — ROC Curve ke neeche ka area; 1.0 = perfect, 0.5 = random
- Hum dono purana (manual) aur naya (RocCurveDisplay) approach dikhate hain

In [ ]:
auc = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

print(f'AUC Score: {auc:.4f}')

In [ ]:
# TARIKA 1 — Manual ROC curve (purana)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Logistic Reg (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Titanic Survival (Manual / Purana Tarika)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# TARIKA 2 — Modern RocCurveDisplay (naya, recommended)
RocCurveDisplay.from_estimator(model, X_test_sc, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.title('ROC Curve — Titanic Survival (RocCurveDisplay / Naya Tarika)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📌 Step 9 — Puri Performance Evaluation ka Summary

**Yeh keywords yaad rakho:**
- **Specificity** = TN / (TN + FP) — model kitna accha deaths identify karta hai
- **Balanced Accuracy** — Sensitivity (Recall) aur Specificity ka average
- **Log Loss** — training ke dauran actually jo minimize hota hai woh value

In [ ]:
from sklearn.metrics import log_loss, balanced_accuracy_score

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print('=' * 55)
print('     TITANIC LOGISTIC REGRESSION — PERFORMANCE')
print('=' * 55)
print(f'  True Positives  (TP) : {tp}')
print(f'  True Negatives  (TN) : {tn}')
print(f'  False Positives (FP) : {fp}')
print(f'  False Negatives (FN) : {fn}')
print('-' * 55)
print(f'  Accuracy            : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Balanced Accuracy   : {balanced_accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision           : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall (Sensitivity): {recall_score(y_test, y_pred):.4f}')
print(f'  Specificity         : {specificity:.4f}')
print(f'  F1 Score            : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC             : {auc:.4f}')
print(f'  Log Loss            : {log_loss(y_test, y_pred_prob):.4f}')
print('=' * 55)

---
## 📌 Step 10 — Multiclass: Passenger Class (pclass) Predict Karna

**Yeh keywords yaad rakho:**
- Ab hum `pclass` (1st, 2nd, 3rd class) predict karenge — yeh 3-class problem hai
- **multi_class='multinomial'** — softmax use karta hai teeno classes ke liye probabilities output karne ke liye
- **macro avg** — saari classes ko equally treat karta hai regardless of size
- **weighted avg** — class imbalance ka dhyan rakhta hai

In [ ]:
# Passenger class predict karne ke liye features
features_mc = ['sex', 'age', 'sibsp', 'parch', 'fare', 'survived']
target_mc   = 'pclass'

df_mc = df[features_mc + [target_mc]].copy()
df_mc['age'].fillna(df_mc['age'].median(), inplace=True)
df_mc['sex'] = LabelEncoder().fit_transform(df_mc['sex'])

X_mc = df_mc.drop('pclass', axis=1)
y_mc = df_mc['pclass']

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc
)

sc_mc = StandardScaler()
X_train_mc_sc = sc_mc.fit_transform(X_train_mc)
X_test_mc_sc  = sc_mc.transform(X_test_mc)

multi_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
multi_model.fit(X_train_mc_sc, y_train_mc)

y_pred_mc = multi_model.predict(X_test_mc_sc)

print('Multiclass Classification Report — Passenger Class Predict Karna:')
print(classification_report(y_test_mc, y_pred_mc, target_names=['1st Class', '2nd Class', '3rd Class']))

In [ ]:
# 3-class ke liye confusion matrix
ConfusionMatrixDisplay.from_estimator(
    multi_model, X_test_mc_sc, y_test_mc,
    display_labels=['1st Class', '2nd Class', '3rd Class'],
    cmap='Blues'
)
plt.title('Multiclass Confusion Matrix — Passenger Class Prediction')
plt.tight_layout()
plt.show()

---
## 🏁 Step 11 — Capstone Exercise

### Students ke liye Challenge

Titanic dataset use karke **sabse accha possible** logistic regression model banao.

**Tasks:**
1. Ek nayi feature banao: `family_size = sibsp + parch + 1`
2. Ek aur feature banao: `is_alone = 1 agar family_size == 1 ho toh, warna 0`
3. `C` ki alag alag values try karo (jaise 0.01, 0.1, 1.0, 10) aur accuracy compare karo
4. Saare metrics report karo: Accuracy, Precision, Recall, F1, AUC
5. Confusion matrix aur ROC curve plot karo

**Shuru karne ke liye template code:**

In [ ]:
# YAHAN APNA CODE LIKHO — Capstone Solution Template
# Step 1: Feature Engineering (nayi features banana)
df_cap = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']].copy()
df_cap['age'].fillna(df_cap['age'].median(), inplace=True)
df_cap['embarked'].fillna(df_cap['embarked'].mode()[0], inplace=True)
df_cap['sex'] = LabelEncoder().fit_transform(df_cap['sex'])
df_cap['embarked'] = LabelEncoder().fit_transform(df_cap['embarked'])

# NAYI FEATURES
df_cap['family_size'] = df_cap['sibsp'] + df_cap['parch'] + 1
df_cap['is_alone']    = (df_cap['family_size'] == 1).astype(int)

print('Nayi DataFrame head:')
print(df_cap.head())

# Step 2: Train/Test split
X_cap = df_cap.drop('survived', axis=1)
y_cap = df_cap['survived']

X_train_cap, X_test_cap, y_train_cap, y_test_cap = train_test_split(
    X_cap, y_cap, test_size=0.2, random_state=42, stratify=y_cap
)

sc_cap = StandardScaler()
X_train_cap_sc = sc_cap.fit_transform(X_train_cap)
X_test_cap_sc  = sc_cap.transform(X_test_cap)

In [ ]:
# Step 3: Alag alag C values try karo
print(f'{'C value':>10} | {'Accuracy':>10} | {'F1':>8} | {'AUC':>8}')
print('-' * 45)

for c_val in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(C=c_val, max_iter=1000, random_state=42)
    m.fit(X_train_cap_sc, y_train_cap)
    yp  = m.predict(X_test_cap_sc)
    ypp = m.predict_proba(X_test_cap_sc)[:, 1]
    print(f'{c_val:>10} | {accuracy_score(y_test_cap, yp):>10.4f} | {f1_score(y_test_cap, yp):>8.4f} | {roc_auc_score(y_test_cap, ypp):>8.4f}')

In [ ]:
# Step 4: Final best model
best_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
best_model.fit(X_train_cap_sc, y_train_cap)

y_pred_cap  = best_model.predict(X_test_cap_sc)
y_proba_cap = best_model.predict_proba(X_test_cap_sc)[:, 1]

print('=== CAPSTONE FINAL MODEL KE RESULTS ===')
print(classification_report(y_test_cap, y_pred_cap, target_names=['Nahi Bacha', 'Bacha']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm_cap = confusion_matrix(y_test_cap, y_pred_cap)
ConfusionMatrixDisplay(cm_cap, display_labels=['Nahi Bacha', 'Bacha']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix — Capstone Model')

# ROC curve
fpr_c, tpr_c, _ = roc_curve(y_test_cap, y_proba_cap)
auc_c = roc_auc_score(y_test_cap, y_proba_cap)
axes[1].plot(fpr_c, tpr_c, color='darkorange', lw=2, label=f'AUC = {auc_c:.2f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — Capstone Model')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✅ Summary — Real Data Pe Kya Apply Kiya

| Step | Kya Kiya |
|---|---|
| Data Loading | Titanic dataset load kiya (891 rows, real-world messy data) |
| EDA | Survival rates visualise kiye — sex, class, age, fare ke hisaab se |
| Preprocessing | Missing age impute kiya, sex aur embarked encode kiya |
| Train-Test Split | 80/20 split, stratified taaki class balance bani rahe |
| Feature Scaling | StandardScaler train aur test dono pe |
| Model Training | LogisticRegression with lbfgs solver |
| Confusion Matrix | seaborn heatmap bhi, ConfusionMatrixDisplay bhi |
| Precision/Recall/F1 | Pura classification report |
| ROC / AUC | Manual plot bhi, RocCurveDisplay bhi |
| Multiclass | Passenger class predict kiya (3 categories) |
| Capstone | Feature engineering + hyperparameter tuning |

---
*Aage: Theory aur formulas ke liye Google Doc padho →*